# SimCLR Batch-Size Ablation (Nidhi)

This notebook runs the batch-size ablation for our SimCLR project.

**What it does:**
- Clones the project repo from GitHub
- Trains SimCLR with batch sizes 64, 128, 256, 512 (short schedule: 20 epochs, 20k images)
- Evaluates each with a linear probe
- Saves all results to `results/results.csv`
- Downloads the results CSV at the end

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → Run all

In [ ]:
# Step 1: Check GPU
# nvidia-smi lists the GPU Colab gave us (it should say Tesla T4) and its memory.
# GPU = graphics card; training a neural network on it is many times faster than on a CPU.
!nvidia-smi

In [ ]:
# Step 2: Clone the repo
# os lets Python work with folders and files (create, delete, change directory).
import os

# Address of our GitHub repository, and the folder in Colab where the code will be copied.
REPO_URL = "https://github.com/ShubhamKang/SimCLR-Deep-Learning.git"
REPO_DIR = "/content/SimCLR-Deep-Learning"

# If an old copy of the code is already there (e.g. from an earlier run), delete it
# so we always start from the latest version on GitHub.
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

# Download (clone) the code from GitHub into REPO_DIR.
!git clone {REPO_URL} {REPO_DIR}
# Move into the project folder, so commands like python train_simclr.py find the files.
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Step 3: Install dependencies
# pip installs Python libraries (-q = quiet output):
#   torch (PyTorch, the deep-learning library), torchvision (image datasets, ResNet-18,
#   augmentations), tqdm (progress bars), pandas (tables), matplotlib (charts).
!pip install -q torch torchvision tqdm pandas matplotlib

In [ ]:
# Step 4: Check everything is in place
# List the files in the project folder, to check the clone worked.
!ls

In [ ]:
# Step 5: Run batch-size ablation
# 20 epochs, 20k image subset — same as augmentation ablation for fair comparison
# Runs 4 pretrainings + 4 linear probe evaluations back to back
# Total time: roughly 25-45 minutes on a T4 GPU

# time is used to measure how long each run takes.
import time

# Fixed short schedule, identical to the augmentation ablation:
# EPOCHS = number of epochs (one epoch = one full pass over the training images).
EPOCHS = 20
# SUBSET = use only 20,000 of the 100,000 unlabelled images, so each run takes minutes.
SUBSET = 20000
# Folder where STL-10 is downloaded (about 2.6 GB, only the first time).
DATA_DIR = "/content/data"

# Create the folders for the data, the checkpoints (saved weights) and the results.
# exist_ok=True means: no error if the folder already exists.
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results", exist_ok=True)

timing = {}  # record time per batch size

# Try each batch size (batch size = how many images are processed together in one
# training step). In SimCLR a bigger batch also means more negatives per image.
for bs in [64, 128, 256, 512]:
    print(f"\n{'='*60}")
    print(f"BATCH SIZE: {bs}")
    print(f"{'='*60}")

    # Remember the start time of this run.
    t_start = time.time()

    # Pretrain
    # Step A: SimCLR pretraining with this batch size (train_simclr.py), on the 20,000-image
    # subset for 20 epochs, all five augmentations. It saves the encoder to checkpoints/bs_<bs>.pt.
    # --workers 2 = two CPU processes prepare the augmented images (Colab has 2 CPU cores).
    ret = os.system(
        f"python train_simclr.py "
        f"--name bs_{bs} "
        f"--epochs {EPOCHS} "
        f"--subset {SUBSET} "
        f"--batch_size {bs} "
        f"--data {DATA_DIR} "
        f"--workers 2"
    )
    # os.system returns 0 if the command succeeded; anything else means it failed,
    # so we print an error and stop the loop.
    if ret != 0:
        print(f"ERROR: pretraining failed for bs={bs}")
        break

    # How many seconds the SimCLR pretraining took.
    t_pretrain = time.time() - t_start

    # Linear probe evaluation
    # Step B: linear probe (evaluate.py --mode linear): freeze the pretrained encoder and train
    # only one linear layer on 100% of the labels; the test accuracy is added to results/results.csv.
    ret = os.system(
        f"python evaluate.py "
        f"--mode linear "
        f"--ckpt checkpoints/bs_{bs}.pt "
        f"--frac 1.0 "
        f"--tag bs_{bs} "
        f"--data {DATA_DIR}"
    )
    # os.system returns 0 if the command succeeded; anything else means it failed,
    # so we print an error and stop the loop.
    if ret != 0:
        print(f"ERROR: evaluation failed for bs={bs}")
        break

    # Total seconds for this batch size (pretraining + linear probe),
    # stored in the timing dictionary and printed in minutes.
    t_total = time.time() - t_start
    timing[bs] = {"pretrain_sec": int(t_pretrain), "total_sec": int(t_total)}
    print(f"bs={bs} done | pretrain: {t_pretrain/60:.1f} min | total: {t_total/60:.1f} min")

# After all batch sizes: print the time of every run (used for Table 4 of the report).
print("\n=== TIMING SUMMARY ===")
for bs, t in timing.items():
    print(f"bs={bs}: pretrain={t['pretrain_sec']//60}m{t['pretrain_sec']%60}s, total={t['total_sec']//60}m{t['total_sec']%60}s")

In [ ]:
# Step 6: Check GPU memory info (note this down for the report)
# Print the GPU name and its memory (total / used / free) to note in the report.
!nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv

In [ ]:
# Step 7: Show results
# pandas (pd) reads the results CSV file as a table (a DataFrame).
import pandas as pd

# Every evaluate.py run added one row to results/results.csv.
df = pd.read_csv("results/results.csv")
# Keep only the batch-size rows (their tag starts with bs_, e.g. bs_128).
bs_results = df[df["tag"].str.startswith("bs_")].copy()
# Read the batch size from the tag: "bs_128"[3:] = "128" -> the number 128.
bs_results["batch_size"] = bs_results["tag"].str[3:].astype(int)
# Sort from the smallest to the largest batch size.
bs_results = bs_results.sort_values("batch_size")
# Convert accuracy from a fraction (0.5811) to a percentage (58.11).
bs_results["test_acc_%"] = (bs_results["test_acc"].astype(float) * 100).round(2)
# Show the final table: batch size and linear-probe test accuracy (%).
print(bs_results[["batch_size", "test_acc_%"]].to_string(index=False))

In [ ]:
# Step 8: Download results.csv to your computer
# This is the file you will commit back to the repo
# Colab's files helper lets us download a file from Colab to our own computer.
from google.colab import files
files.download("results/results.csv")

## After this notebook finishes

1. The `results.csv` download will start automatically in Step 8.
2. Copy the downloaded file into your local repo at `results/results.csv` (it already has Sandhya's supervised baseline rows — this adds your bs_* rows).
3. Note down the timing numbers from Step 5 and the GPU name from Step 6 — you'll need them for the batch-size analysis section of the report.
4. Come back here and tell me the results — I'll write the analysis section.